# 🏪 Zava Agentic Fine-Tuning Lab — 06: Wrap-Up

Congratulations on completing the lab! Here's a summary of what you've learned
and where to go next.

---
## What We Learned

### 1. RFT Improves Agentic Tool-Calling Tasks

The model learned to apply policy rules more accurately **without being shown the "right" answer**.
Through trial and error (reinforcement learning), it discovered how to:
- Call `get_order` reliably to fetch order data
- Apply the correct return window based on loyalty tier and product category
- Compute restocking fees accurately
- Handle edge cases (defective + sale → store credit, late delivery credits, etc.)

### 2. The Grader Is Everything

**Partial credit** (not binary pass/fail) is critical for learning.
The model needs gradient signal to learn incrementally:

```
score = 0.4 × action_correct + 0.3 × amount_correct + 0.2 × policy_reasoning + 0.1 × tool_usage
```

A model that gets the action right but the amount wrong scores 0.5 — much more
informative than a binary 0.

### 3. Calibrate Your Pass Threshold

Target **30–50% failure rate** on the base model:
- Too easy = no learning signal (model already passes)
- Too hard = sparse reward (model can't find the right direction)
- We used **0.80** which gave ~35% failure rate — ideal for RL

### 4. Always Evaluate Checkpoints

The best checkpoint isn't always the last one!
Training metrics (reward) don't perfectly predict real-world performance.


| Model | Avg Score | P@0.9 | P@0.8 |
|-------|-----------|-------|-------|
| Base o4-mini | 73.2% | 37% | 60% |
| Step 95 (peak train reward) | 93.7% | 88% | 88% |
| Step 115 (best eval) | 97.9% | 100% | 100% |

### 5. Start Small, Iterate Fast

We validated with 60 examples first, then scaled to 343.
Don't invest in a large dataset until the grader and threshold are working.

---
## Lab Recap — What You Did

| Notebook | What You Did |
|----------|-------------|
| **01 – Introduction & Setup** | Connected to Microsoft Foundry, verified environment |
| **02 – Meet the Agent** | Ran the Zava agent live on 3 scenarios, saw tool calling in action |
| **03 – Baseline & Grader** | Evaluated base o4-mini (~73%), understood the grader and threshold calibration |
| **04 – Build Data & Submit** | Explored training data format, built a custom example, submitted an RFT job |
| **05 – Results & Evaluate** | Visualized reward curves, compared checkpoints, ran head-to-head evaluation (~87%) |
| **06 – Wrap-Up** | Reviewed key takeaways and next steps (this notebook) |

---
## What to Try Next

### Check Your Submitted Job
Your RFT job from notebook 04 will finish in 8-10 hours. Check the reward curve!

### Experiment Ideas

| Experiment | What to Change | Expected Effect |
|-----------|---------------|----------------|
| **Tighter threshold** | `pass_threshold=0.85` or `0.90` | Stricter policy compliance, slower convergence |
| **More training data** | Generate additional scenario variations | Better generalization to unseen cases |
| **Lower learning rate** | `learning_rate_multiplier=0.5` | Smoother convergence, less forgetting |
| **More epochs** | `n_epochs=5` | More training, risk of overfitting |
| **SFT comparison** | Use a large model's outputs as training data | Compare RFT vs SFT distillation |

---
## Check Your Job Status

Run the cell below to check the status of your submitted RFT job.
You'll need the job ID from notebook 04.

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential = DefaultAzureCredential()
)
client = project_client.get_openai_client(api_key=os.environ["API_KEY"])

# Replace with your job ID from notebook 04
JOB_ID = "<paste-your-job-id-here>"

job = client.fine_tuning.jobs.retrieve(JOB_ID)
print(f"Job: {job.id}")
print(f"Status: {job.status}")
print(f"Model: {job.model}")
if job.fine_tuned_model:
    print(f"\n🎉 Fine-tuned model ready: {job.fine_tuned_model}")
elif job.status == "failed":
    print(f"\n❌ Job failed. Check the Microsoft Foundry portal for details.")
else:
    print(f"\n⏳ Still running... check back later.")

Replace JOB_ID with your actual job ID from notebook 04.
You can also check status in the Microsoft Foundry portal.


---
## Resources

- [Azure AI Fine-Tuning docs](https://learn.microsoft.com/azure/ai-services/openai/how-to/fine-tuning)
- [RFT with graders guide](https://developers.openai.com/api/docs/guides/graders)
- [Agentic RFT with tools](https://developers.openai.com/api/docs/guides/reinforcement-fine-tuning)

---

### 🎉 Thank you for completing the Zava Agentic Fine-Tuning Lab!

You've seen how RFT can improve an AI agent's accuracy on complex tool-calling tasks
from **~73% to ~87%** — without ever showing the model a single "correct" response.